In [10]:
!uv pip install rich openpyxl

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 6 packages in 389ms                                         
Installed 2 packages in 6ms                                 
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5


# Check metadata and register subjects in entitycore

Requirements:
- [entitysdk](https://github.com/openbraininstitute/entitysdk) v0.7.2 or higher!!
- rich
- openpyxl

Subject: **FlyWire FAFB** — the adult female *Drosophila melanogaster* (FAFB EM volume, Zheng et al. 2018) reconstructed by FlyWire into the connectome underlying the computational brain model of Shiu, Sterne et al. (2024, Nature; DOI: [10.1038/s41586-024-07763-9](https://www.nature.com/articles/s41586-024-07763-9)).

Reads from the local spreadsheet `Drosophila_brain_model_subject.xlsx`.

In [1]:
import numpy as np
import pandas as pd
import os
import shutil
import time
import obi_one as obi

from conntility import ConnectivityMatrix
from datetime import datetime, timedelta
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from pathlib import Path
from rich import print as rprint

In [5]:
environment = "staging" 
token = get_token(environment=environment)
# Staging shared project + tests
project_context = ProjectContext(virtual_lab_id="e6030ed8-a589-4be2-80a6-f975406eb1f6", project_id="2720f785-a3a2-4472-969d-19a53891c817")
client = Client(environment=environment, project_context=project_context, token_manager=token)

# environment = "production"
# token = get_token(environment=environment)
# project_context = ProjectContext.from_vlab_url("https://www.openbraininstitute.org/app/virtual-lab/lab/5f8376bf-b84f-4188-8ef5-e1df3d7529b4/project/7d22829c-edc6-4b1d-8ab9-99dd9e511e74/home")
# client = Client(environment=environment, project_context=project_context, token_manager=token)

In [6]:
if environment == "staging":
    ID_key = "entityCoreIDStaging"
else:
    ID_key = "entityCoreID"

In [7]:
DAYS_PER_YEAR = 365.2425  # Mean length of the Gregorian calendar year

## List of subjects to be registered

In [8]:
# Spreadsheet with the subject(s) to be registered.
# Subject backing DOI 10.1038/s41586-024-07763-9 (FlyWire FAFB specimen).
subjects_file = "Drosophila_brain_model_subject.xlsx"  # relative to this notebook's folder

In [11]:
subjects_df = pd.read_excel(subjects_file, sheet_name="subjects")
subjects_df

,entityCoreID,entityCoreIDStaging,species,subjectName,subjectDescription,subjectAge,subjectAgeUnit,subjectAgePeriod,subjectSex
0,NaN,NaN,Drosophila melanogaster,FlyWire FAFB,Single adult female Drosophila melanogaster wh...,7,days,postnatal,female


#### Find or create subjects

In [12]:
make_public = False
check_only = True  # Dry-run by default; set to False to actually register

In [13]:
entity_df = subjects_df[[]].copy()
for idx, subject_row in subjects_df.iterrows():
    if not str(subject_row[ID_key]) == "nan":
        continue  # Skip if ID already in table

    # Find species
    species = client.search_entity(
        entity_type=models.Species, query={"name__ilike": subject_row["species"]}
    ).first()
    
    assert species is not None, f"ERROR: Species '{subject_row['species']}' not found!"

    # Find subject, if already existing
    subjects = client.search_entity(
        entity_type=models.Subject, query={"name": subject_row["subjectName"]}
    ).all()

    if len(subjects) > 0:
        # Check consistency and use existing subject
        assert len(subjects) == 1, f"ERROR: Multiple subject entities with name '{subject_row['subjectName']}' found!"
        subject = subjects[0]
        assert subject.description == subject_row["subjectDescription"], "ERROR: Subject description mismatch!"
        assert subject.sex == subject_row["subjectSex"], "ERROR: Subject description mismatch!"
        assert subject.species.id == species.id, "ERROR: Subject species mismatch!"
        if np.isfinite(subject_row["subjectAge"]):
            age = subject_row["subjectAge"]
            age_unit = subject_row["subjectAgeUnit"]
            age_value = timedelta(**{age_unit: age})
            assert subject.age_value == age_value
        else:
            assert subject.age_value is None
            assert subject.age_min is None
            assert subject.age_max is None
        if isinstance(subject_row["subjectAgePeriod"], str):
            assert subject.age_period == subject_row["subjectAgePeriod"]
        else:
            assert subject.age_period is None
        print("Using existing subject:", end=" ")
    else:
        # Register new subject
        age_dict = {}
        if np.isfinite(subject_row["subjectAge"]):
            age = subject_row["subjectAge"]
            age_unit = subject_row["subjectAgeUnit"]
            if age_unit == "years":  # Not supported by timedelta, so convert to days
                age = int(np.round(age * DAYS_PER_YEAR))
                age_unit = "days"
            age_dict["age_value"] = timedelta(**{age_unit: age})
        if isinstance(subject_row["subjectAgePeriod"], str):
            age_dict["age_period"] = subject_row["subjectAgePeriod"]
        subject = models.Subject(
            name=subject_row["subjectName"],
            description=subject_row["subjectDescription"],
            authorized_public=make_public,
            sex=subject_row["subjectSex"],
            species=species,
            **age_dict
        )
        if check_only:
            print("CHECK ONLY:", end=" ")
        else:
            subject = client.register_entity(subject)
            print("Registered new subject:", end=" ")
    print(f"{subject.name}, ID {subject.id}")
    entity_df.loc[idx, "name"] = subject.name
    entity_df.loc[idx, ID_key] = subject.id

CHECK ONLY: FlyWire FAFB, ID None


In [14]:
# List of registered subjects
entity_df

,name,entityCoreIDStaging
0,FlyWire FAFB,None


In [16]:
# Export to .csv file
csv_file = f"registered_subjects_{environment}_" + str(datetime.now()).replace(" ", "_").replace(":", "-") + ".csv"
entity_df.to_csv(csv_file)